# 📈 Linear Regression & Gradient Descent

This notebook covers:
- **Simple Linear Regression** from scratch using NumPy
- **Gradient Descent** implemented manually to understand the optimization process
- **Multiple Linear Regression** with scikit-learn
- **Visualization** of cost function convergence, regression lines, and residuals
- **Model Evaluation** using MSE, RMSE, MAE, and R² Score

> 🎯 **Goal:** Understand how a model learns by minimizing a cost function using Gradient Descent.

---
## 📦 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.datasets import fetch_california_housing
import warnings
warnings.filterwarnings('ignore')

# Set consistent plot style
sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)

print('✅ All libraries imported successfully!')
print(f'NumPy: {np.__version__}, Pandas: {pd.__version__}')

**Expected Output:**
```
✅ All libraries imported successfully!
NumPy: 1.x.x, Pandas: 2.x.x
```

---
## 🔢 2. Simple Linear Regression — From Scratch

### What is Linear Regression?
Linear Regression models the relationship between input **X** and output **y** as:

$$\hat{y} = \theta_0 + \theta_1 \cdot X$$

Where:
- $\theta_0$ = **bias** (intercept)
- $\theta_1$ = **weight** (slope)

We want to find $\theta_0$ and $\theta_1$ that **minimize the cost function** (Mean Squared Error).

In [ ]:
# --------------------------------------------------
# Generate synthetic dataset: y = 3 + 2*X + noise
# True slope = 2, True intercept = 3
# --------------------------------------------------
X = 2 * np.random.rand(100, 1)              # 100 samples, feature values in [0, 2]
y = 3 + 2 * X + np.random.randn(100, 1)    # Linear relationship + Gaussian noise

# Plot the raw data
plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.7, color='steelblue', edgecolors='white', s=60, label='Data points')
plt.xlabel('X (Feature)', fontsize=13)
plt.ylabel('y (Target)', fontsize=13)
plt.title('Synthetic Dataset: y = 3 + 2X + noise', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

print(f'Dataset shape: X={X.shape}, y={y.shape}')
print(f'X range: [{X.min():.2f}, {X.max():.2f}]')
print(f'y range: [{y.min():.2f}, {y.max():.2f}]')

**Expected Output:**
```
Dataset shape: X=(100, 1), y=(100, 1)
X range: [0.00, 2.00]
y range: [2.xx, 8.xx]
```
A scatter plot showing a clear upward linear trend with slight noise.

---
## ⚙️ 3. Gradient Descent — Manual Implementation

### The Cost Function (MSE):
$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2$$

### Update Rules:
$$\theta_1 := \theta_1 - \alpha \cdot \frac{\partial J}{\partial \theta_1}, \quad \theta_0 := \theta_0 - \alpha \cdot \frac{\partial J}{\partial \theta_0}$$

Where $\alpha$ is the **learning rate** — controls how big a step we take.

In [ ]:
# --------------------------------------------------
# Gradient Descent Implementation From Scratch
# --------------------------------------------------

def compute_cost(X, y, theta):
    """
    Compute Mean Squared Error cost.
    J = (1/2m) * sum( (X @ theta - y)^2 )
    """
    m = len(y)
    predictions = X.dot(theta)
    cost = (1 / (2 * m)) * np.sum((predictions - y) ** 2)
    return cost


def gradient_descent(X, y, learning_rate=0.1, n_iterations=1000):
    """
    Performs Batch Gradient Descent.
    
    Parameters:
        X            : Feature matrix (with bias column added)
        y            : Target vector
        learning_rate: Step size alpha
        n_iterations : Number of iterations to run
    
    Returns:
        theta        : Learned parameters [theta_0, theta_1]
        cost_history : MSE at each iteration for plotting
    """
    m = len(y)
    theta = np.zeros((X.shape[1], 1))   # Initialize weights to zero
    cost_history = []

    for i in range(n_iterations):
        predictions = X.dot(theta)              # Forward pass: ŷ = Xθ
        errors = predictions - y                # Residuals: ŷ - y
        gradients = (1 / m) * X.T.dot(errors)  # Gradient: (1/m) * Xᵀ(ŷ - y)
        theta -= learning_rate * gradients      # Parameter update step
        cost_history.append(compute_cost(X, y, theta))

    return theta, cost_history


# Add bias column (column of 1s) to X for the intercept term θ₀
X_b = np.c_[np.ones((100, 1)), X]   # X_b shape: (100, 2)

# Run Gradient Descent
theta_gd, cost_history = gradient_descent(X_b, y, learning_rate=0.1, n_iterations=1000)

print('🎯 Gradient Descent Results:')
print(f'  Learned intercept (θ₀): {theta_gd[0][0]:.4f}  (True: 3.0)')
print(f'  Learned slope     (θ₁): {theta_gd[1][0]:.4f}  (True: 2.0)')
print(f'  Final Cost (MSE/2): {cost_history[-1]:.6f}')

**Expected Output:**
```
🎯 Gradient Descent Results:
  Learned intercept (θ₀): 3.01xx  (True: 3.0)
  Learned slope     (θ₁): 1.99xx  (True: 2.0)
  Final Cost (MSE/2): 0.xxxxxx
```
The learned parameters should be very close to the true values (3 and 2).

In [ ]:
# --------------------------------------------------
# Visualize: Cost Convergence + Regression Line
# --------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Cost Function over Iterations ---
axes[0].plot(cost_history, color='crimson', linewidth=2)
axes[0].set_xlabel('Iterations', fontsize=12)
axes[0].set_ylabel('Cost J(θ)', fontsize=12)
axes[0].set_title('Gradient Descent: Cost Convergence', fontsize=13)
axes[0].axhline(y=cost_history[-1], color='gray', linestyle='--', alpha=0.5, label=f'Min Cost: {cost_history[-1]:.4f}')
axes[0].legend()

# --- Right: Final Regression Line vs. Data ---
X_new = np.array([[0], [2]])                        # Two points to draw the line
X_new_b = np.c_[np.ones((2, 1)), X_new]            # Add bias column
y_predict = X_new_b.dot(theta_gd)                  # Predicted values

axes[1].scatter(X, y, alpha=0.6, color='steelblue', edgecolors='white', s=60, label='Data')
axes[1].plot(X_new, y_predict, 'r-', linewidth=2.5, label=f'GD Line: y={theta_gd[0][0]:.2f}+{theta_gd[1][0]:.2f}x')
axes[1].set_xlabel('X', fontsize=12)
axes[1].set_ylabel('y', fontsize=12)
axes[1].set_title('Learned Regression Line (Gradient Descent)', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.show()

**Expected Output:**
- **Left plot:** Cost starts high and rapidly decreases, flattening out after ~100–200 iterations — this confirms convergence.
- **Right plot:** A red regression line tightly fitting through the blue scatter points.

---
## 🔍 4. Effect of Learning Rate on Convergence

The **learning rate (α)** is a critical hyperparameter:
- **Too large** → cost may oscillate or diverge
- **Too small** → convergence is very slow
- **Just right** → smooth, fast convergence

In [ ]:
# --------------------------------------------------
# Compare multiple learning rates
# --------------------------------------------------

learning_rates = [0.001, 0.01, 0.1, 0.5]
colors = ['blue', 'green', 'orange', 'red']

plt.figure(figsize=(10, 6))

for lr, color in zip(learning_rates, colors):
    _, costs = gradient_descent(X_b, y, learning_rate=lr, n_iterations=200)
    # Cap costs to avoid huge values from divergence
    costs_clipped = np.clip(costs, 0, 30)
    plt.plot(costs_clipped, color=color, linewidth=2, label=f'α = {lr}')

plt.xlabel('Iterations', fontsize=12)
plt.ylabel('Cost J(θ) [clipped at 30]', fontsize=12)
plt.title('Effect of Learning Rate on Gradient Descent Convergence', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('Observations:')
print('  α=0.001 → Very slow convergence, needs many more iterations')
print('  α=0.01  → Moderate speed, converges smoothly')
print('  α=0.1   → Fast convergence (optimal zone)')
print('  α=0.5   → May overshoot/diverge depending on data')

**Expected Output:**
- 4 curves showing different convergence speeds
- Small α → slow decay; Large α → quick drop or instability

---
## 🏠 5. Multiple Linear Regression — California Housing Dataset

Now we apply Linear Regression to a real-world dataset with **multiple features**.

In [ ]:
# --------------------------------------------------
# Load and explore the California Housing dataset
# --------------------------------------------------

housing = fetch_california_housing(as_frame=True)
df = housing.frame

print('📊 Dataset Info:')
print(f'  Shape: {df.shape}  ({df.shape[0]} samples, {df.shape[1]-1} features + 1 target)')
print(f'  Features: {list(housing.feature_names)}')
print(f'  Target: MedHouseVal (Median house value in $100,000s)\n')

print('📈 Statistical Summary:')
display(df.describe().round(2))

**Expected Output:**
```
📊 Dataset Info:
  Shape: (20640, 9)  (20640 samples, 8 features + 1 target)
  Features: ['MedInc', 'HouseAge', 'AveRooms', ...]
  Target: MedHouseVal
```
Followed by a statistical summary table with count, mean, std, min, max for each column.

In [ ]:
# --------------------------------------------------
# Correlation heatmap to understand feature relationships
# --------------------------------------------------

plt.figure(figsize=(10, 7))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Upper triangle mask
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', vmin=-1, vmax=1,
    linewidths=0.5, square=True, cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap — California Housing', fontsize=14)
plt.tight_layout()
plt.show()

# Show top correlations with the target
print('\n🔗 Top correlations with MedHouseVal (target):')
print(corr_matrix['MedHouseVal'].sort_values(ascending=False).to_string())

**Expected Output:**
- A triangular heatmap showing correlation coefficients between all features
- **MedInc** will have the highest positive correlation with the target (~0.69)

In [ ]:
# --------------------------------------------------
# Prepare data: split + scale
# --------------------------------------------------

X_house = df.drop('MedHouseVal', axis=1)
y_house = df['MedHouseVal']

# 80% train, 20% test split
X_train, X_test, y_train, y_test = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

# Feature scaling (important for gradient-based methods)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Fit on train, transform train
X_test_scaled  = scaler.transform(X_test)         # Transform test with same scaler

print(f'Train set: {X_train.shape}  |  Test set: {X_test.shape}')
print(f'Train mean (after scaling): {X_train_scaled.mean():.6f}  (should be ~0)')
print(f'Train std  (after scaling): {X_train_scaled.std():.6f}   (should be ~1)')

**Expected Output:**
```
Train set: (16512, 8)  |  Test set: (4128, 8)
Train mean (after scaling): 0.000000  (should be ~0)
Train std  (after scaling): 1.000000  (should be ~1)
```

In [ ]:
# --------------------------------------------------
# Train LinearRegression (closed-form Normal Equation)
# and SGDRegressor (Stochastic Gradient Descent)
# --------------------------------------------------

# --- Model 1: sklearn LinearRegression (uses Normal Equation internally) ---
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

# --- Model 2: SGDRegressor (online/stochastic gradient descent) ---
sgd_model = SGDRegressor(max_iter=1000, learning_rate='optimal', random_state=42)
sgd_model.fit(X_train_scaled, y_train)
y_pred_sgd = sgd_model.predict(X_test_scaled)


def evaluate_model(name, y_true, y_pred):
    """Print MSE, RMSE, MAE, and R² for a regression model."""
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f'\n📊 {name}:')
    print(f'   MSE  : {mse:.4f}')
    print(f'   RMSE : {rmse:.4f}  (in $100,000s)')
    print(f'   MAE  : {mae:.4f}')
    print(f'   R²   : {r2:.4f}  (1.0 = perfect fit)')


evaluate_model('Linear Regression (Normal Equation)', y_test, y_pred_lr)
evaluate_model('SGD Regressor (Stochastic GD)', y_test, y_pred_sgd)

**Expected Output:**
```
📊 Linear Regression (Normal Equation):
   MSE  : 0.5559
   RMSE : 0.7456  (in $100,000s)
   MAE  : 0.5332
   R²   : 0.5758  (1.0 = perfect fit)

📊 SGD Regressor (Stochastic GD):
   MSE  : ~0.56xx
   R²   : ~0.57xx
```
Both models should produce very similar metrics since they solve the same problem.

In [ ]:
# --------------------------------------------------
# Visualize: Predictions vs Actual & Residuals
# --------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Predictions vs Actual ---
axes[0].scatter(y_test, y_pred_lr, alpha=0.3, color='steelblue', s=20)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Values', fontsize=12)
axes[0].set_ylabel('Predicted Values', fontsize=12)
axes[0].set_title('Actual vs Predicted (Linear Regression)', fontsize=13)
axes[0].legend()

# --- Residual Plot ---
residuals = y_test.values - y_pred_lr
axes[1].scatter(y_pred_lr, residuals, alpha=0.3, color='coral', s=20)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted Values', fontsize=12)
axes[1].set_ylabel('Residuals (Actual - Predicted)', fontsize=12)
axes[1].set_title('Residual Plot', fontsize=13)

plt.tight_layout()
plt.show()

print('📌 Interpretation:')
print('  - Predictions vs Actual: Points along the red dashed line = perfect predictions')
print('  - Residual Plot: Random scatter around 0 = good model (no systematic bias)')

**Expected Output:**
- **Left:** Points clustered around the red diagonal line (perfect = on the line)
- **Right:** Residuals roughly centered at 0, randomly scattered (some pattern visible due to model limitations)

In [ ]:
# --------------------------------------------------
# Feature Importance: Inspect model coefficients
# Higher absolute coefficient → stronger influence on prediction
# --------------------------------------------------

coef_df = pd.DataFrame({
    'Feature': X_house.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', ascending=False)

plt.figure(figsize=(9, 5))
colors_bar = ['green' if c > 0 else 'crimson' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_bar, edgecolor='white')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Coefficient Value (Standardized)', fontsize=12)
plt.title('Linear Regression — Feature Coefficients', fontsize=13)
plt.tight_layout()
plt.show()

print('\nFeature Coefficients (standardized):')
print(coef_df.to_string(index=False))

**Expected Output:**
- **MedInc** (Median Income) will have the largest positive coefficient — the strongest predictor of house price
- **Latitude** will have a large negative coefficient — northern California has lower prices

---
## 🧮 6. Normal Equation vs Gradient Descent — Comparison

| Method | Formula | Pros | Cons |
|---|---|---|---|
| Normal Equation | $\theta = (X^TX)^{-1}X^Ty$ | Exact solution, no learning rate | Slow for large $n$ features (matrix inversion) |
| Batch GD | Iterative update | Works for large datasets | Needs learning rate tuning |
| SGD | One sample per step | Very fast, online learning | Noisy updates |
| Mini-batch GD | Small batch per step | Balance of speed + stability | Batch size tuning needed |

In [ ]:
# --------------------------------------------------
# Normal Equation — Closed-form solution
# θ = (XᵀX)⁻¹ Xᵀy   — directly solves for optimal θ
# --------------------------------------------------

theta_normal = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)

print('🧮 Normal Equation Results (on synthetic dataset):')
print(f'  Intercept (θ₀): {theta_normal[0][0]:.4f}  (True: 3.0)')
print(f'  Slope     (θ₁): {theta_normal[1][0]:.4f}  (True: 2.0)')

print('\n📊 Comparison — Gradient Descent vs Normal Equation:')
print(f'  GD  → θ₀: {theta_gd[0][0]:.4f}, θ₁: {theta_gd[1][0]:.4f}')
print(f'  NE  → θ₀: {theta_normal[0][0]:.4f}, θ₁: {theta_normal[1][0]:.4f}')
print('\n✅ Both methods converge to nearly identical solutions!')

**Expected Output:**
```
🧮 Normal Equation Results:
  Intercept (θ₀): 3.01xx  (True: 3.0)
  Slope     (θ₁): 1.99xx  (True: 2.0)

📊 Comparison:
  GD → θ₀: 3.01xx, θ₁: 1.99xx
  NE → θ₀: 3.01xx, θ₁: 1.99xx
✅ Both methods converge to nearly identical solutions!
```

---
## ✅ 7. Summary

| Concept | Key Takeaway |
|---|---|
| **Linear Regression** | Models target as weighted sum of features |
| **Cost Function (MSE)** | Measures how far predictions are from truth |
| **Gradient Descent** | Iteratively reduces cost by adjusting θ in direction of steepest descent |
| **Learning Rate** | Controls step size; too high = divergence, too low = slow convergence |
| **Normal Equation** | Analytical solution; no iterations needed but computationally expensive for large data |
| **R² Score** | Proportion of variance explained by the model (1.0 = perfect) |
| **Feature Scaling** | Critical for gradient-based optimization to converge efficiently |

---
*Next Notebook → Classification Algorithms & Model Tuning* 🚀